In [16]:
import uuid
from pathlib import Path 

import cv2
import numpy as np 
import pandas as pd
from tqdm import tqdm
from deepface import DeepFace 
from deepface.modules import preprocessing
from deepface.models.facial_recognition.Facenet import load_facenet128d_model
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import Distance, VectorParams, PointStruct

In [3]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)

In [5]:
CLIENT.recreate_collection(collection_name='Headshots_512',
                           vectors_config=VectorParams(size=512, distance=Distance.COSINE))

True

In [12]:
headshot_dir = Path('../../data/headshots')
headshots = [x for x in headshot_dir.iterdir()]
for headshot in tqdm(headshots):
    face = cv2.imread(str(headshot))
    name, cast_id, num = headshot.stem.split('_')
    encoding = DeepFace.represent(face, 
                                    model_name='Facenet512', 
                                    detector_backend='retinaface', 
                                    enforce_detection=True,
                                    normalization='Facenet2018',
                                    max_faces=1,
                                    align=True)[0]['embedding']
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        payload={'name': name,
                'tmdb_id': cast_id},
        vector=encoding
    )
    CLIENT.upsert(collection_name='Headshots_512',
                  points=[point])
    

100%|██████████| 38/38 [00:31<00:00,  1.22it/s]


In [13]:
headshot_dir = Path('../../data/headshots')
headshots = [x for x in headshot_dir.iterdir()]
for headshot in tqdm(headshots):
    face = cv2.imread(str(headshot))
    name, cast_id, num = headshot.stem.split('_')
    encoding = DeepFace.represent(face, 
                                    model_name='Facenet', 
                                    detector_backend='retinaface', 
                                    enforce_detection=True,
                                    normalization='Facenet',
                                    max_faces=1,
                                    align=True)[0]['embedding']
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        payload={'name': name,
                'tmdb_id': cast_id},
        vector=encoding
    )
    CLIENT.upsert(collection_name='Headshots_128',
                  points=[point])

100%|██████████| 38/38 [00:12<00:00,  3.06it/s]


In [15]:
CLIENT.recreate_collection(collection_name='Headshots_Custom',
                           vectors_config=VectorParams(size=128, distance=Distance.COSINE))

True

In [17]:
model = load_facenet128d_model()

In [18]:
headshot_dir = Path('../../data/headshots')
headshots = [x for x in headshot_dir.iterdir()]
for headshot in tqdm(headshots):
    face = cv2.imread(str(headshot))
    name, cast_id, num = headshot.stem.split('_')
    f = preprocessing.normalize_input(preprocessing.resize_image(face, (160, 160)), normalization='Facenet')
    encoding = model(f)[0]
    # encoding = DeepFace.represent(face, 
    #                                 model_name='Facenet', 
    #                                 detector_backend='retinaface', 
    #                                 enforce_detection=True,
    #                                 normalization='Facenet',
    #                                 max_faces=1,
    #                                 align=True)[0]['embedding']
    point = models.PointStruct(
        id=str(uuid.uuid4()),
        payload={'name': name,
                'tmdb_id': cast_id},
        vector=encoding._numpy()
    )
    CLIENT.upsert(collection_name='Headshots_Custom',
                  points=[point])

100%|██████████| 38/38 [00:12<00:00,  2.97it/s]
